# Telecom Fraud Detection

**call_id**                        - Unique identifier for each call record (e.g. CALL0000001)

**caller_age_days**                - How long the caller's number/account has
existed, in days. Fraudulent numbers tend to be newer.

**calls_per_day**                 - Average number of calls made by this caller per day. Spammers/fraudsters often call at high volume.

**call_duration_sec**             - Duration of this specific call, in seconds.

**avg_call_duration_sec**          - This caller's average call duration across all their calls.

**unique_receivers_24h**           - Number of distinct people this caller contacted in the last 24 hours. A high count can indicate mass dialing/spam.

**receiver_block_rate**            - Percentage of receivers who have blocked this caller.

**spam_reports_count**             - How many times this caller has been reported as spam.

**country_code_risk_score**        - Risk score assigned to the caller's country/region code, based on historical fraud rates from that region.

**night_call_ratio**               - Proportion of this caller's calls made during night hours. Unusual night-calling patterns can be a fraud signal.

**sequential_dialing_score**       - Measures how "sequential" or automated the calling pattern looks (e.g. dialing numbers in order), typical of robocalls/scripted fraud.

**graph_degree**                   - Number of connections (distinct numbers/people) this caller has in the call network.

**previous_fraud_associations**    - Count of past connections to numbers/accounts previously flagged as fraudulent.

**reputation_score**               - Overall trust/reputation score for the caller (higher = more trustworthy).

**fraud_label**                    - Target variable: 1 = fraudulent call, 0 = legitimate call.

## What changed from the original notebook

1. **Imputation instead of `dropna()`** — the old code dropped ~51% of rows because nulls were scattered across every column. We now only drop rows where the *target* (`fraud_label`) is missing, and fill missing feature values with the column median.
2. **Outlier capping (IQR method)** — the dataset has injected extreme outliers (15–25x normal values). These were distorting `StandardScaler` and hurting the model. We clip values to the IQR-based bounds before training.
3. **Random Forest instead of Logistic Regression** — tree-based models handle non-linear relationships and are naturally robust to outliers/scale, which fits this feature set (scores, ratios, counts) much better.
4. **`class_weight='balanced'`** — accounts for the ~69/31 class split so the model doesn't get lazy and just predict "not fraud."
5. **Proper metrics** — accuracy alone is misleading on imbalanced data. We now report precision, recall, F1, and ROC-AUC, plus 5-fold cross-validation so the score isn't a fluke of one split.
6. **Feature importance plot** — helps you explain *why* the model predicts fraud, not just that it does.

In [ ]:
#import libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

In [ ]:
#data set

dataset = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ML/Telecom Fraud Detection - Project/telecom_fraud_detection_dataset_impure.csv')
print(dataset.shape)
dataset.head()

In [ ]:
# drop the ID column - it's not a predictive feature, just a row identifier
dataset.drop('call_id', axis=1, inplace=True)
print(dataset.shape)

In [ ]:
print(dataset.info())
print(dataset.isnull().sum())

## Data cleaning

**Why not `dropna()`?** Nulls are spread across every column (~4-8% each), so dropping any row
with a null wipes out roughly half the dataset. Instead:
- Drop rows only where the **target** (`fraud_label`) itself is missing — those rows are unusable no matter what.
- Impute missing **feature** values with the column median (robust to outliers, simple, works well as a baseline).

In [ ]:
# drop duplicate rows
before = dataset.shape[0]
dataset.drop_duplicates(inplace=True)
print(f"Dropped {before - dataset.shape[0]} duplicate rows -> {dataset.shape}")

In [ ]:
# drop rows where the target itself is missing (can't train/evaluate on these)
before = dataset.shape[0]
dataset.dropna(subset=['fraud_label'], inplace=True)
dataset['fraud_label'] = dataset['fraud_label'].astype(int)
print(f"Dropped {before - dataset.shape[0]} rows with missing label -> {dataset.shape}")

In [ ]:
# impute missing feature values with the column median
feature_cols = dataset.columns.drop('fraud_label')

for col in feature_cols:
    n_missing = dataset[col].isnull().sum()
    if n_missing > 0:
        dataset[col] = dataset[col].fillna(dataset[col].median())

print("Remaining nulls:", dataset.isnull().sum().sum())

## Outlier handling

The dataset has injected extreme outliers (some values 15-25x normal). We cap them using the
IQR method rather than dropping the rows, so we don't lose potentially-fraudulent signal
(fraud cases can legitimately be extreme, e.g. very high `calls_per_day`) — we just stop them
from blowing up the scale.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['calls_per_day', 'graph_degree', 'sequential_dialing_score']):
    sns.boxplot(y=dataset[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# cap outliers to the IQR bounds (winsorizing) instead of deleting rows
for col in feature_cols:
    Q1 = dataset[col].quantile(0.25)
    Q3 = dataset[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    dataset[col] = dataset[col].clip(lower, upper)

print("Outliers capped.")

In [ ]:
#features and target

X = dataset[feature_cols]
y = dataset['fraud_label']

print("Class balance:")
print(y.value_counts(normalize=True))

In [ ]:
# train test split (stratify keeps the same fraud/non-fraud ratio in both sets)
x_train, x_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2, stratify=y
)

## Model: Random Forest

Switched from Logistic Regression to Random Forest:
- Handles non-linear feature interactions (e.g. `sequential_dialing_score` combined with `graph_degree`)
- Robust to outliers and differing feature scales — no `StandardScaler` needed
- `class_weight='balanced'` compensates for the ~69/31 class split

Feel free to swap in `GradientBoostingClassifier` or `xgboost.XGBClassifier` later for a further boost —
Random Forest is a strong, low-effort upgrade to start with.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model.fit(x_train, y_train)

In [ ]:
#y_predict on test and train
y_pred_test = model.predict(x_test)
y_pred_train = model.predict(x_train)

## Evaluation

Accuracy alone can be misleading with imbalanced classes. Precision/recall/F1 tell you how well
the model actually catches fraud (recall) vs. how often its fraud calls are correct (precision).

In [ ]:
print("Train Accuracy:", model.score(x_train, y_train))
print("Test Accuracy :", accuracy_score(y_test, y_pred_test))
print("Precision     :", precision_score(y_test, y_pred_test))
print("Recall        :", recall_score(y_test, y_pred_test))
print("F1 Score      :", f1_score(y_test, y_pred_test))
print("ROC-AUC       :", roc_auc_score(y_test, model.predict_proba(x_test)[:, 1]))
print()
print(classification_report(y_test, y_pred_test, target_names=['Not Fraud', 'Fraud']))

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
ConfusionMatrixDisplay(cm, display_labels=['Not Fraud', 'Fraud']).plot(cmap='magma')
plt.show()

## Cross-validation

A single train/test split can get lucky or unlucky. 5-fold CV gives a more reliable estimate
of how the model performs on unseen data.

In [ ]:
cv_scores = cross_val_score(model, X, y, cv=5, scoring='f1')
print("CV F1 scores:", cv_scores)
print("Mean F1:", cv_scores.mean(), "  Std:", cv_scores.std())

## Comparing multiple models

Random Forest is a strong choice, but it's worth comparing it against a few other classifiers
to confirm it's actually the best fit for this data — the same approach used in the AI-Student-Impact
notebook (Logistic Regression, Decision Tree, Random Forest, SVM, KNN), adapted for this binary
fraud-detection task.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Logistic Regression, SVM and KNN are distance/gradient based, so they need scaled features.
# Tree-based models (Decision Tree, Random Forest) don't need scaling, but scaling doesn't hurt them either.
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
def evaluate_model(name, y_actual, y_pred):
    """Prints accuracy/precision/recall/F1 and returns them as a dict for comparison."""
    acc = accuracy_score(y_actual, y_pred)
    prec = precision_score(y_actual, y_pred)
    rec = recall_score(y_actual, y_pred)
    f1 = f1_score(y_actual, y_pred)

    print(f"--- {name} ---")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1 score :", f1)
    print(classification_report(y_actual, y_pred, target_names=['Not Fraud', 'Fraud']))
    print()

    return {"Model": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}

results = []

# keep the Random Forest result from earlier
results.append(evaluate_model("Random Forest", y_test, y_pred_test))

In [ ]:
# Logistic Regression
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(x_train_scaled, y_train)
log_reg_pred = log_reg.predict(x_test_scaled)

results.append(evaluate_model("Logistic Regression", y_test, log_reg_pred))

In [ ]:
# Decision Tree
decision_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=8,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42
)
decision_tree.fit(x_train, y_train)
tree_pred = decision_tree.predict(x_test)

results.append(evaluate_model("Decision Tree", y_test, tree_pred))

In [ ]:
# Support Vector Machine
svm_model = SVC(
    C=1.0,
    kernel="rbf",
    gamma="scale",
    probability=True,
    class_weight='balanced',
    random_state=42
)
svm_model.fit(x_train_scaled, y_train)
svm_pred = svm_model.predict(x_test_scaled)

results.append(evaluate_model("SVM (RBF)", y_test, svm_pred))

In [ ]:
# K-Nearest Neighbors
knn = KNeighborsClassifier(
    n_neighbors=7,
    weights='distance',
    metric='minkowski',
    n_jobs=-1
)
knn.fit(x_train_scaled, y_train)
knn_pred = knn.predict(x_test_scaled)

results.append(evaluate_model("KNN", y_test, knn_pred))

## Model comparison

Side-by-side view of all five models on the same test set, so you can justify picking one
over the others (and cite the numbers in your project writeup).

In [ ]:
results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
results_df

In [ ]:
plt.figure(figsize=(9, 5))
results_melted = results_df.melt(id_vars='Model', value_vars=['Accuracy', 'Precision', 'Recall', 'F1'])
sns.barplot(data=results_melted, x='Model', y='value', hue='variable', palette='magma')
plt.title('Model Comparison')
plt.ylim(0, 1)
plt.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

**Note on Random Forest vs the rest:** tree-based ensemble models tend to win on tabular data like
this because the features are mostly scores/ratios/counts with non-linear thresholds that matter
for fraud (e.g. `spam_reports_count` past a certain point). Logistic Regression and SVM assume more
linear separability, and KNN struggles a bit in higher-dimensional feature spaces. If Random Forest
comes out on top here, that's expected — but it's good practice to show the comparison rather than
just asserting it.

In [ ]:
sns.heatmap(dataset.corr(), annot=True, cmap='magma', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

## Feature importance

Which features actually drive the model's fraud predictions?

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, palette='magma', legend=False)
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

importances

In [ ]:
import pickle

# Save the best-performing model based on the comparison above
best_model_name = results_df.iloc[0]['Model']
model_lookup = {
    "Random Forest": model,
    "Logistic Regression": log_reg,
    "Decision Tree": decision_tree,
    "SVM (RBF)": svm_model,
    "KNN": knn
}
best_model = model_lookup[best_model_name]
print("Best model:", best_model_name)

with open("model_pickle.pkl", "wb") as f:
    pickle.dump((best_model, scaler, best_model_name), f)

In [ ]:
import pickle

with open("model_pickle.pkl", "rb") as f:
    model = pickle.load(f)